# Tugas 2 - Prediksi sisa umur mesin

Eksperimen regresi RUL menggunakan jendela data mesin dan split per mesin. Notebook menghasilkan prediksi, interval, serta bagian laporan.

## Sel 1 · Masukan dan keluaran

Dataset RUL dibaca dari folder UTS. Ubah DATA bila letak Drive berbeda. HASIL menampung prediksi, statistik kalibrasi, dan bukti eksperimen.

In [ ]:
import os
from pathlib import Path

try:
    from google.colab import drive
except ImportError:
    drive = None

if drive is not None:
    if "DATA" not in globals() and not Path("/content/drive/MyDrive").is_dir():
        drive.mount("/content/drive")
    dasar_data = "/content/drive/MyDrive/UTS_DL/data_uts_dl"
    dasar_hasil = "/content/hasil_tugas2"
else:
    dasar_data = "data_uts_dl"
    dasar_hasil = "hasil_tugas2"

DATA = Path(globals().get("DATA", os.environ.get("UTS_DATA_DIR", dasar_data))).resolve()
HASIL = Path(globals().get("HASIL", os.environ.get("UTS_HASIL_TUGAS2", dasar_hasil))).resolve()
HASIL.mkdir(parents=True, exist_ok=True)
BUKTI = HASIL / "artifacts" / "tugas2"
BAGIAN = HASIL / "laporan_bagian" / "02_prediksi_rul.pdf"

if not DATA.is_dir():
    raise FileNotFoundError(f"Periksa lokasi dataset: {DATA}")
print("Data:", DATA, "| Keluaran:", HASIL)

## Sel 2 · Fungsi bersama dan perangkat

Fungsi bersama mengatur seed, mencatat sidik SHA-256 dataset, dan menyimpan artefak. CPU tiga thread merupakan konfigurasi referensi.

In [ ]:
%pip -q install reportlab pypdf

"""Pemuatan data, seed dan penyimpanan keluaran."""
import hashlib
import importlib.metadata
import json
import random
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch

SEED = (17, 29, 43)


def atur_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True


def baca_pasangan(folder, nama, id_kolom, modalitas):
    meta = pd.read_csv(folder / f"{nama}_meta.csv").sort_values("indeks").reset_index(drop=True)
    with np.load(folder / f"{nama}.npz", allow_pickle=False) as arsip:
        gambar, kedua = arsip["gambar"], arsip[modalitas]
    if not (len(meta) == len(gambar) == len(kedua)):
        raise ValueError(f"Panjang metadata dan array berbeda pada {nama}")
    if not meta[id_kolom].is_unique or not np.array_equal(meta.indeks, np.arange(len(meta))):
        raise ValueError(f"ID atau indeks tidak valid pada {nama}")
    return meta, gambar, kedua


def sidik(folder, nama):
    hasil = {}
    for file in nama:
        hash_file = hashlib.sha256()
        with (folder / file).open("rb") as stream:
            for bagian in iter(lambda: stream.read(1024 * 1024), b""):
                hash_file.update(bagian)
        hasil[file] = hash_file.hexdigest()
    return hasil


def versi():
    paket = ("torch", "numpy", "pandas", "scipy", "scikit-learn", "matplotlib", "reportlab", "threadpoolctl")
    return {"Python": sys.version.split()[0], **{p: importlib.metadata.version(p) for p in paket}}


def simpan_json(path, isi):
    temp = path.with_suffix(".tmp")
    temp.write_text(json.dumps(isi, ensure_ascii=False, allow_nan=False, indent=2), encoding="utf-8")
    temp.replace(path)


def simpan_model(path, isi):
    temp = path.with_suffix(".tmp")
    torch.save(isi, temp)
    temp.replace(path)

DEVICE = torch.device(os.environ.get('UTS_DEVICE', 'cpu') if torch.cuda.is_available() else 'cpu')
if DEVICE.type == 'cpu':
    torch.set_num_threads(3)
print('Perangkat:', DEVICE)

## Sel 3 · Normalisasi kausal, GRU, CNN, dan kalibrasi

RUL dihitung dari siklus gagal dengan batas 150. Setelan membentuk enam rezim operasi. Normalisasi median dan MAD dipelajari hanya dari mesin latih. Jendela hingga 40 siklus dibandingkan melalui baseline boosting, GRU, dan CNN. Model utama dipilih dari RMSE dev dan interval dikalibrasi pada mesin terpisah.

In [ ]:
"""Prediksi RUL dari riwayat mesin, dengan evaluasi per mesin."""
import json
import math

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from sklearn.cluster import KMeans
from sklearn.ensemble import HistGradientBoostingRegressor
from torch import nn
from torch.nn import functional as F


SENSOR = [f"s{i:02}" for i in range(1, 22)]
SETEL = ["setel_1", "setel_2", "setel_3"]
PANJANG = 40
BERKAS = ("rul_train.csv.gz", "rul_test.csv.gz")


def baca(path):
    data = pd.read_csv(path).sort_values(["id_unit", "siklus"]).reset_index(drop=True)
    if data.duplicated(["id_unit", "siklus"]).any():
        raise ValueError("Satu mesin memiliki baris ganda pada siklus yang sama")
    if data[SETEL].isna().any().any():
        raise ValueError("Konfigurasi operasi tidak lengkap")
    return data


def sesuaikan(data):
    raw_set = data[SETEL].to_numpy()
    pusat_set, skala_set = raw_set.mean(0), raw_set.std(0).clip(1e-6)
    z = (raw_set - pusat_set) / skala_set
    grup = KMeans(6, random_state=2026, n_init=8).fit(z[::5])
    rezim = grup.predict(z)
    s = data[SENSOR].to_numpy()
    pusat = np.stack([np.nanmedian(s[rezim == i], axis=0) for i in range(6)])
    mad = np.stack([np.nanmedian(np.abs(s[rezim == i] - pusat[i]), axis=0) for i in range(6)])
    fallback = np.nanstd(s, axis=0)
    skala = np.where(mad > 1e-7, 1.4826 * mad, fallback).clip(1e-7)
    return {"pusat_set": pusat_set, "skala_set": skala_set, "pusat_rezim": grup.cluster_centers_,
            "pusat_sensor": pusat, "skala_sensor": skala}


def normalisasi(data, prep):
    setel = (data[SETEL].to_numpy() - prep["pusat_set"]) / prep["skala_set"]
    r = ((setel[:, None, :] - prep["pusat_rezim"]) ** 2).sum(2).argmin(1)
    sensor = (data[SENSOR].to_numpy() - prep["pusat_sensor"][r]) / prep["skala_sensor"][r]
    hilang = np.isnan(sensor)
    spike = np.abs(sensor) > 8
    sensor = pd.DataFrame(sensor).groupby(data.id_unit.to_numpy(), sort=False).ffill().fillna(0)
    umur = data.siklus.to_numpy()[:, None] / 250
    x = np.column_stack([sensor.to_numpy().clip(-8, 8), hilang, setel, umur, np.ones(len(data))])
    return x.astype(np.float32), r, spike


def riwayat(data, prep):
    x, _, _ = normalisasi(data, prep)
    return {int(u): (x[np.asarray(i)], data.siklus.to_numpy()[np.asarray(i)])
            for u, i in data.groupby("id_unit", sort=True).indices.items()}


def jendela(histori, akhir=None, stride=8, label=True):
    X, y, meta = [], [], []
    for u, (x, siklus) in histori.items():
        if akhir is None:
            ujung = sorted(set(range(0, len(x), stride)) | {len(x)-1})
        else:
            ujung = [akhir[u]]
        for j in ujung:
            potong = x[max(j-PANJANG+1, 0):j+1]
            alas = np.zeros((PANJANG, x.shape[1]), dtype=np.float32)
            alas[-len(potong):] = potong
            X.append(alas)
            meta.append((u, int(siklus[j])))
            if label:
                y.append(min(150, siklus[-1] - siklus[j]))
    return np.stack(X), np.array(y, dtype=np.float32), meta


def ringkas(x):
    valid = x[:, :, -1:]
    n = valid.sum(1).clip(1)
    mean = x.sum(1) / n
    dev = (((x - mean[:, None]) ** 2 * valid).sum(1) / n).clip(0) ** 0.5
    baru = x[:, -10:].sum(1) / valid[:, -10:].sum(1).clip(1)
    return np.column_stack((x[:, -1], mean, dev, baru - mean))


class Mesin(nn.Module):
    def __init__(self, dim, jenis):
        super().__init__()
        self.jenis = jenis
        self.proyeksi = nn.Linear(dim, 32)
        if jenis == "gru":
            self.encoder = nn.GRU(32, 48, batch_first=True)
            panjang = 48
        else:
            self.encoder = nn.Sequential(nn.Conv1d(32, 48, 5, padding=2), nn.SiLU(),
                                         nn.Conv1d(48, 48, 5, dilation=2, padding=4), nn.SiLU(),
                                         nn.Conv1d(48, 48, 5, dilation=4, padding=8), nn.SiLU())
            panjang = 96
        self.kepala = nn.Sequential(nn.Linear(panjang + 2 * dim, 64), nn.LayerNorm(64),
                                    nn.SiLU(), nn.Dropout(.12), nn.Linear(64, 1))

    def forward(self, x):
        valid = x[:, :, -1:]
        n = valid.sum(1).clamp_min(1)
        fitur = F.silu(self.proyeksi(x)) * valid
        if self.jenis == "gru":
            _, h = self.encoder(fitur)
            v = h[-1]
        else:
            h = self.encoder(fitur.transpose(1, 2)).transpose(1, 2)
            v = torch.cat([h[:, -1], (h * valid).sum(1) / n], dim=1)
        return self.kepala(torch.cat([v, x[:, -1], x.sum(1) / n], dim=1)).squeeze(1)


@torch.inference_mode()
def prediksi(model, x):
    model.eval()
    return np.concatenate([(model(b) * 150).clamp(0, 150).cpu().numpy() for b in x.split(256)])


def metrik(y, pred):
    d = np.asarray(pred, dtype=np.float64) - y
    return {"n": len(y), "rmse": float(np.sqrt(np.mean(d**2))),
            "asimetris": float(np.expm1(np.where(d < 0, -d / 13, d / 10)).sum())}


def pita(y):
    return {"0-30": y <= 30, "31-60": (y > 30) & (y <= 60), "61-100": (y > 60) & (y <= 100),
            "101-149": (y > 100) & (y < 150), "150": y == 150}


def beda_rentang(y, p):
    return {k: metrik(y[m], p[m]) for k, m in pita(y).items() if m.any()}


def latih(jenis, seed, train, dev, epoch, device, out, identitas):
    atur_seed(seed)
    x, y = train
    xv, yv = dev
    model = Mesin(x.shape[-1], jenis).to(device)
    path = out / f"{jenis}_{seed}.pt"
    config = [identitas, jenis, seed, epoch, str(device), 1]
    if path.is_file():
        c = torch.load(path, map_location="cpu", weights_only=True)
        if c["config"] == config:
            model.load_state_dict(c["model"])
            return model, c["kurva"]
    opt = torch.optim.AdamW(model.parameters(), lr=.002, weight_decay=.001)
    jadwal = torch.optim.lr_scheduler.CosineAnnealingLR(opt, epoch, eta_min=.00022)
    skor, state, kurva = math.inf, None, []
    for j in range(epoch):
        model.train()
        total = 0.0
        for i in torch.randperm(len(x), device=device).split(256):
            loss = F.smooth_l1_loss(model(x[i]), y[i] / 150, beta=.15)
            opt.zero_grad(set_to_none=True)
            loss.backward()
            opt.step()
            total += loss.item() * len(i)
        jadwal.step()
        rmse = metrik(yv, prediksi(model, xv))["rmse"]
        kurva.append({"epoch": j+1, "loss": total / len(x), "rmse_dev": rmse})
        if rmse < skor:
            skor = rmse
            state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        print(f"T2 | {jenis} | seed {seed} | epoch {j+1}/{epoch} | RMSE dev {rmse:.3f}", flush=True)
    simpan_model(path, {"config": config, "model": state, "kurva": kurva})
    model.load_state_dict(state)
    return model, kurva


def visual(out, d, y, p, lo, hi, dipilih, diag):
    fig, ax = plt.subplots(1, 2, figsize=(10, 4), layout="constrained")
    ix = np.arange(len(diag))
    ax[0].bar(ix, diag["hilang"] * 100, label="Hilang")
    ax[0].bar(ix, diag["spike"] * 100, bottom=diag["hilang"] * 100, label="Ekstrem")
    ax[0].set(xticks=ix, xticklabels=SENSOR, ylabel="Persen baris latih")
    ax[0].tick_params(axis="x", labelrotation=75, labelsize=7)
    ax[0].legend(fontsize=8)
    contoh = d.assign(rezim=d.rezim.astype(str)).groupby("rezim")["setel_1"].mean()
    ax[1].bar(contoh.index, contoh.values)
    ax[1].set(xlabel="Rezim operasi", ylabel="Rerata setel_1")
    fig.savefig(out / "eksplorasi.png", dpi=145)
    plt.close(fig)
    fig, ax = plt.subplots(1, 2, figsize=(10, 4), layout="constrained")
    urut = np.argsort(y)
    xx = np.arange(len(y))
    ax[0].fill_between(xx, lo[urut], hi[urut], alpha=.23, color="tab:blue", label="Interval 90%")
    ax[0].plot(xx, y[urut], color="black", lw=1.4, label="RUL aktual")
    ax[0].scatter(xx, p[urut], s=8, label="Prediksi")
    ax[0].set(xlabel="Mesin evaluasi, diurutkan berdasarkan RUL", ylabel="Siklus", ylim=(-5, 155))
    ax[0].legend(fontsize=8)
    for jenis in ("gru", "cnn"):
        df = pd.read_csv(out / f"kurva_{jenis}_{SEED[0]}.csv")
        ax[1].plot(df.epoch, df.rmse_dev, label=f"{jenis.upper()} (seed {SEED[0]})", marker=".")
    ax[1].set(xlabel="Epoch", ylabel="RMSE dev", title=f"Ensemble terpilih: {dipilih.upper()}")
    ax[1].legend(fontsize=8)
    fig.savefig(out / "hasil.png", dpi=145)
    plt.close(fig)


def jalankan(data_dir, out, hasil, device, cepat=False):
    model_dir = out / "model"
    model_dir.mkdir(parents=True, exist_ok=True)
    h = sidik(data_dir, BERKAS)
    train, test = (baca(data_dir / f) for f in BERKAS)
    unit = sorted(train.id_unit.unique())
    if set(unit) & set(test.id_unit):
        raise ValueError("ID mesin latih dan uji bertumpang tindih")
    rng = np.random.default_rng(2026)
    u = rng.permutation(unit)
    a, b, c, d = np.split(u, [int(len(u)*.60), int(len(u)*.72), int(len(u)*.84)])
    if cepat:
        a, b, c, d = a[:65], b[:20], c[:20], d[:23]
    split = dict(zip(("latih", "dev", "kalibrasi", "evaluasi"), (a, b, c, d)))
    data_fit = train[train.id_unit.isin(a)].copy()
    prep = sesuaikan(data_fit)
    np.savez(out / "normalisasi.npz", **prep)
    normal, rezim, spike = normalisasi(data_fit, prep)
    data_fit["rezim"] = rezim
    cap = np.minimum(150, data_fit.groupby("id_unit").siklus.transform("max") - data_fit.siklus)
    diagnostik = pd.DataFrame({"sensor": SENSOR, "hilang": data_fit[SENSOR].isna().mean().to_numpy(),
                              "spike": spike.mean(0), "std_mentah": data_fit[SENSOR].std().to_numpy()})
    diagnostik["spearman_rul"] = pd.DataFrame(normal[:, :len(SENSOR)]).corrwith(
        pd.Series(cap.to_numpy()), method="spearman").fillna(0).to_numpy()
    diagnostik["korelasi_lemah"] = diagnostik.spearman_rul.abs() < .04
    diagnostik.to_csv(out / "diagnostik.csv", index=False)
    median_sensor_rezim = data_fit.groupby("rezim")[SENSOR].median()
    rentang_rezim = median_sensor_rezim.max() - median_sensor_rezim.min()
    renggang = (rentang_rezim / data_fit[SENSOR].std().clip(lower=1e-7)).sort_values(ascending=False)
    histori = {nama: riwayat(train[train.id_unit.isin(ids)], prep) for nama, ids in split.items()}
    akhir = {nama: {j: int(rng.integers(min(39, len(x)-2), len(x)-1)) for j, (x, _) in histori[nama].items()}
             for nama in ("dev", "kalibrasi", "evaluasi")}
    pd.DataFrame([(u, nama, akhir.get(nama, {}).get(u)) for nama, ids in split.items() for u in ids],
                 columns=["id_unit", "split", "indeks_akhir_simulasi"]).to_csv(out / "pembagian.csv", index=False)
    x, y, _ = jendela(histori["latih"])
    xv, yv, mv = jendela(histori["dev"], akhir["dev"])
    xc, yc, mc = jendela(histori["kalibrasi"], akhir["kalibrasi"])
    xe, ye, me = jendela(histori["evaluasi"], akhir["evaluasi"])
    htest = riwayat(test, prep)
    xt, _, mt = jendela(htest, {id_: len(item[0])-1 for id_, item in htest.items()}, label=False)
    # Mengubah siklus setelah endpoint harus mempertahankan fitur input endpoint.
    uji_u, uji_siklus = me[0]
    baris_uji = train[train.id_unit == uji_u]
    berubah = baris_uji.copy()
    berubah.loc[berubah.siklus > uji_siklus, SENSOR] = 99999.0
    x_awal, _, _ = jendela(riwayat(baris_uji, prep), {uji_u: akhir["evaluasi"][uji_u]})
    x_ubah, _, _ = jendela(riwayat(berubah, prep), {uji_u: akhir["evaluasi"][uji_u]})
    np.testing.assert_array_equal(x_awal, x_ubah)
    baseline = HistGradientBoostingRegressor(max_iter=30 if cepat else 180, max_leaf_nodes=24,
                                            l2_regularization=3, learning_rate=.07, random_state=17)
    baseline.fit(ringkas(x), y)
    base_v, base_e = [baseline.predict(ringkas(feat)).clip(0, 150) for feat in (xv, xe)]
    def t(x):
        return torch.as_tensor(x, device=device)
    tx, ty, tv, tc, te, tt = map(t, (x, y, xv, xc, xe, xt))
    epoch = 1 if cepat else 18
    identitas = json.dumps({"hash": h, "cepat": cepat}, sort_keys=True)
    semua, seeds = {}, {}
    banding = {"boosting": {"dev": metrik(yv, base_v), "evaluasi": metrik(ye, base_e),
                           "per_rentang": beda_rentang(ye, base_e)}}
    for jenis in ("gru", "cnn"):
        pp = []
        for seed in SEED:
            net, kurva = latih(jenis, seed, (tx, ty), (tv, yv), epoch, device, model_dir, identitas)
            pd.DataFrame(kurva).to_csv(out / f"kurva_{jenis}_{seed}.csv", index=False)
            pp.append([prediksi(net, q) for q in (tv, tc, te, tt)])
        pv, pc, pe, pt = [np.mean(z, axis=0) for z in zip(*pp)]
        semua[jenis] = (pc, pe, pt)
        seeds[jenis] = [{"seed": s, **metrik(ye, r[2])} for s, r in zip(SEED, pp)]
        banding[jenis] = {"dev": metrik(yv, pv), "evaluasi": metrik(ye, pe),
                         "per_rentang": beda_rentang(ye, pe)}
    final = min(("gru", "cnn"), key=lambda j: banding[j]["dev"]["rmse"])
    pc, pe, pt = semua[final]
    rank = min(len(yc), math.ceil(.90 * (len(yc) + 1)))
    q = float(np.sort(np.abs(yc - pc))[rank-1])
    bawah, atas = (pe-q).clip(0, 150), (pe+q).clip(0, 150)
    kena = (ye >= bawah) & (ye <= atas)
    pred_test = pd.DataFrame(mt, columns=["id_unit", "siklus"]).assign(rul=pt)
    pred_test[["id_unit", "rul"]].to_csv(hasil / "prediksi_rul.csv", index=False, float_format="%.6f")
    pred_test.assign(bawah=(pt-q).clip(0, 150), atas=(pt+q).clip(0, 150),
                     tinjau_servis=(pt-q) <= 30).to_csv(out / "interval_uji.csv", index=False)
    pd.DataFrame(me, columns=["id_unit", "siklus"]).assign(
        aktual=ye, prediksi=pe, bawah=bawah, atas=atas).to_csv(out / "evaluasi.csv", index=False)
    pd.DataFrame(mc, columns=["id_unit", "siklus"]).assign(aktual=yc, prediksi=pc).to_csv(
        out / "kalibrasi.csv", index=False)
    visual(out, data_fit, ye, pe, bawah, atas, final, diagnostik)
    simpan_json(out / "hasil.json", {
        "cepat": cepat, "epoch": epoch, "device": str(device), "versi": versi(), "hash": h,
        "mesin": {k: len(v) for k, v in split.items()}, "mesin_uji": len(mt), "baris": [len(train), len(test)],
        "jendela_latih": len(y), "rezim": 6, "rentang_rezim_std_terbesar": renggang.head(4).to_dict(),
        "sensor_korelasi_lemah": diagnostik.loc[diagnostik.korelasi_lemah, "sensor"].tolist(),
        "hilang": float(data_fit[SENSOR].isna().to_numpy().mean()), "spike": float(spike.mean()),
        "kausal": True, "model": banding, "final": final, "seed": seeds[final],
        "interval": {"nominal": .90, "rank": rank, "q": q, "cakupan": float(kena.mean()),
                     "lebar": float((atas-bawah).mean()), "cakupan_per_rentang": {
                         k: float(kena[m].mean()) for k, m in pita(ye).items() if m.any()}},
    })

## Sel 4 · Bagian laporan dan verifikasi

Laporan mencakup diagnostik sensor, RMSE dan skor asimetris per rentang, serta cakupan interval. Verifikasi menghitung ulang label, metrik evaluasi, dan cakupan, serta mencocokkan seluruh ID prediksi.

In [ ]:
"""Laporan Bahasa Indonesia, dirakit dari hasil eksperimen terverifikasi."""
from xml.sax.saxutils import escape

from reportlab.lib import colors
from reportlab.lib.enums import TA_LEFT
from reportlab.lib.pagesizes import A4
from reportlab.lib.styles import ParagraphStyle
from reportlab.platypus import Image, PageBreak, Paragraph, SimpleDocTemplate, Spacer, Table, TableStyle



BIRU = colors.HexColor("#173452")
TEAL = colors.HexColor("#166254")
TEKS = colors.HexColor("#202b36")
PUCAT = colors.HexColor("#edf4f7")
ABU = colors.HexColor("#546476")
W = 505
STYLE = {
    "kicker": ParagraphStyle("kicker", fontName="Helvetica-Bold", fontSize=9, leading=12,
                             textColor=TEAL, spaceAfter=8),
    "h1": ParagraphStyle("h1", fontName="Helvetica-Bold", fontSize=22, leading=25,
                         textColor=BIRU, spaceAfter=11),
    "h2": ParagraphStyle("h2", fontName="Helvetica-Bold", fontSize=12, leading=15,
                         textColor=BIRU, spaceBefore=9, spaceAfter=5),
    "p": ParagraphStyle("p", fontName="Helvetica", fontSize=9.4, leading=13.7,
                        textColor=TEKS, alignment=TA_LEFT, spaceAfter=7),
    "caption": ParagraphStyle("caption", fontName="Helvetica", fontSize=8.2, leading=11.5,
                              textColor=ABU, spaceBefore=4, spaceAfter=8),
    "cell": ParagraphStyle("cell", fontName="Helvetica", fontSize=8.4, leading=10.7, textColor=TEKS),
    "label": ParagraphStyle("label", fontName="Helvetica-Bold", fontSize=8.4, leading=10.7,
                            textColor=colors.white),
}


def angka(x, digit=3):
    return f"{x:,.{digit}f}".replace(",", "~").replace(".", ",").replace("~", ".")


def n(x):
    return angka(x, 0)


def pct(x, digit=1):
    return angka(x * 100, digit) + "%"


def footer(canvas, doc):
    canvas.setTitle("UTS Deep Learning | Klasifikasi, RUL, dan Klaster")
    canvas.setAuthor("Kelompok UTS Deep Learning")
    canvas.setStrokeColor(colors.HexColor("#d8e3e9"))
    canvas.line(45, 40, 550, 40)
    canvas.setFillColor(ABU)
    canvas.setFont("Helvetica", 8)
    canvas.drawString(45, 27, "UTS DEEP LEARNING  /  EKSPERIMEN MULTIMODAL")
    canvas.drawRightString(550, 27, f"T2 / {doc.page}")


from pypdf import PdfReader


def buat_bagian(out, r, target):
    art = out / 'artifacts'
    story = []
    def p(s, gaya="p"):
        story.append(Paragraph(s, STYLE[gaya]))

    def judul(bagian, h):
        p(bagian, "kicker")
        p(h, "h1")

    def sub(s):
        p(s, "h2")

    def tabel(header, rows, lebar):
        isi = [[Paragraph(escape(str(s)), STYLE["label"]) for s in header]]
        isi += [[Paragraph(escape(str(s)), STYLE["cell"]) for s in row] for row in rows]
        t = Table(isi, colWidths=lebar, repeatRows=1, hAlign="LEFT")
        t.setStyle(TableStyle([("BACKGROUND", (0, 0), (-1, 0), BIRU),
                               ("ROWBACKGROUNDS", (0, 1), (-1, -1), [PUCAT, colors.white]),
                               ("VALIGN", (0, 0), (-1, -1), "TOP"),
                               ("LEFTPADDING", (0, 0), (-1, -1), 7),
                               ("RIGHTPADDING", (0, 0), (-1, -1), 7),
                               ("TOPPADDING", (0, 0), (-1, -1), 5),
                               ("BOTTOMPADDING", (0, 0), (-1, -1), 5)]))
        story.append(t)
        story.append(Spacer(1, 7))

    def figur(tugas, nama, hmax, ket):
        im = Image(str(art / f"tugas{tugas}" / f"{nama}.png"))
        skala = min(W / im.imageWidth, hmax / im.imageHeight)
        im.drawWidth, im.drawHeight = im.imageWidth*skala, im.imageHeight*skala
        im.hAlign = "CENTER"
        story.append(im)
        p(ket, "caption")

    def halaman():
        story.append(PageBreak())

    rt = r["model"][r["final"]]["evaluasi"]
    judul("TUGAS 2 / DATA", "Riwayat operasi dan label sisa umur")
    p(f"Data berisi {n(r['baris'][0])} siklus latih dan {n(r['baris'][1])} siklus uji. "
      f"Prediksi dibuat hanya pada siklus terakhir dari {n(r['mesin_uji'])} mesin uji. Label latih "
      "dihitung sebagai min(150, siklus gagal - siklus saat ini). Nilai 150 menyatakan sisa umur setidaknya 150 siklus.")
    tabel(["Kelompok", "Mesin", "Penggunaan"], [
        [nama.title(), n(r["mesin"][nama]), ket] for nama, ket in (
            ("latih", "Normalisasi, fitting baseline dan model neural"),
            ("dev", "Pemilihan epoch dan arsitektur"),
            ("kalibrasi", "Kuantil galat absolut split conformal"),
            ("evaluasi", "RMSE, skor asimetris dan cakupan akhir"),
        )
    ], [98, 56, 351])
    sub("Rezim, sensor, dan data hilang")
    rentang = ", ".join(f"{j} ({angka(v, 2)} SD)" for j, v in r["rentang_rezim_std_terbesar"].items())
    lemah = ", ".join(r["sensor_korelasi_lemah"]) or "tidak ada"
    p(f"K-means pada tiga setelan terstandardisasi memodelkan {r['rezim']} rezim dari mesin latih. "
      f"Sensor dengan rentang median antar rezim terbesar relatif terhadap simpangan baku mentah: {rentang}. "
      f"Proporsi bacaan hilang {pct(r['hilang'])} dan proporsi ekstrem {pct(r['spike'], 3)} pada latih. "
      f"Sensor dengan |Spearman| &lt; 0,04 terhadap RUL setelah normalisasi: {lemah}. "
      "Semua sensor dipertahankan karena korelasi marginal tidak menilai informasi temporal atau interaksi sensor.")
    p("Per rezim, sensor dikurangi median dan dibagi 1,4826 MAD. Skala hampir nol diganti simpangan baku "
      "global sensor latih. Bacaan hilang diisi residual terakhir yang tersedia pada mesin yang sama "
      "(mulai dari nol bila sejarah kosong). Mask hilang dipertahankan dan residual dibatasi ke [-8, 8].")
    figur(2, "eksplorasi", 211, "Bacaan hilang dan di luar delapan satuan robust per sensor. "
          "Panel kanan menampilkan salah satu perbedaan setelan antar rezim.")
    sub("Jendela kausal")
    p(f"Jendela 40 siklus dipotong di endpoint, memakai padding kiri dan indikator validitas. "
      f"Stride delapan dan endpoint kegagalan menghasilkan {n(r['jendela_latih'])} jendela latih. "
      "Setelan dan umur mesin ditambahkan sebagai fitur. Setiap mesin dev, kalibrasi dan evaluasi "
      "memiliki satu titik berhenti pra-kegagalan yang disimulasikan. Mengubah sensor setelah "
      "endpoint evaluasi pertama tidak mengubah tensor masukan pada pemeriksaan kausal.")

    halaman()
    judul("TUGAS 2 / MODEL", "Perbandingan, galat, dan interval")
    p("Baseline histogram gradient boosting memakai nilai terbaru, rata-rata, simpangan baku, dan "
      "perubahan rata-rata sepuluh siklus terakhir. GRU membaca urutan dan memakai state terakhir. "
      "CNN 1D berdilasi memakai fitur akhir serta rata-rata dengan mask. Keduanya memakai Smooth L1 "
      f"pada RUL berskala, sampai {r['epoch']} epoch, dan rata-rata tiga prediksi seed.")
    tabel(["Model", "RMSE dev", "RMSE evaluasi", "Skor asimetris evaluasi"], [
        [("Boosting" if j == "boosting" else j.upper()), angka(v["dev"]["rmse"], 2),
         angka(v["evaluasi"]["rmse"], 2), angka(v["evaluasi"]["asimetris"], 2)]
        for j, v in r["model"].items()
    ], [113, 113, 114, 165])
    rr = r["model"][r["final"]]["per_rentang"]
    interval = r["interval"]
    tabel(["RUL aktual", "Mesin", "RMSE", "Skor asimetris", "Cakupan"], [
        [a, n(v["n"]), angka(v["rmse"], 2), angka(v["asimetris"], 2),
         pct(interval["cakupan_per_rentang"][a])] for a, v in rr.items()
    ], [100, 65, 85, 135, 120])
    p("Untuk d = prediksi - aktual, kontribusi skor adalah exp(-d/13) - 1 bila d negatif, "
      "dan exp(d/10) - 1 bila d nonnegatif. Skor menjumlahkan kontribusi tiap mesin, sehingga "
      "perbandingan antar rentang dipengaruhi ukuran kelompok. Prediksi optimistis dihukum lebih besar.")
    dominan = max(rr, key=lambda j: rr[j]["asimetris"])
    p(f"Rentang {dominan} menyumbang {pct(rr[dominan]['asimetris'] / (rt['asimetris'] or 1))} "
      f"skor asimetris model utama. Cakupan interval pada rentang itu {pct(interval['cakupan_per_rentang'][dominan])}.")
    p(f"Ensemble <b>{r['final'].upper()}</b> dipilih dari RMSE dev. Interval split conformal nominal 90% "
      f"memakai galat absolut rank {interval['rank']} dari mesin kalibrasi, q = {angka(interval['q'], 2)} siklus. "
      f"Setelah pemotongan ke [0, 150], cakupan evaluasi {pct(interval['cakupan'])} "
      f"dengan lebar rata-rata {angka(interval['lebar'], 2)} siklus.")
    figur(2, "hasil", 203, "RUL dan interval pada mesin evaluasi. Kurva dev memperlihatkan hasil seed 17.")
    p("Tinjau servis lebih awal jika batas bawah interval paling banyak 30 siklus. Ambang ini merupakan "
      "contoh kebijakan triase. Cakupan berlaku secara marginal dengan asumsi pertukaran mesin "
      "kalibrasi dan evaluasi. Distribusi penghentian data uji yang berbeda bisa menurunkan cakupan.")
    doc = SimpleDocTemplate(str(target), pagesize=A4, leftMargin=45, rightMargin=45,
                            topMargin=40, bottomMargin=56)
    doc.build(story, onFirstPage=footer, onLaterPages=footer)

def verifikasi_tugas2(ringkas, pdf):
    pred = pd.read_csv(HASIL / "prediksi_rul.csv")
    test = pd.read_csv(DATA / "rul_test.csv.gz", usecols=["id_unit"])
    assert list(pred.columns) == ["id_unit", "rul"]
    assert pred.id_unit.is_unique and not pred.isna().any().any()
    assert set(pred.id_unit) == set(test.id_unit)
    assert pred.rul.between(0, 150).all()

    ev = pd.read_csv(BUKTI / "evaluasi.csv")
    split = pd.read_csv(BUKTI / "pembagian.csv")
    assert split.id_unit.is_unique
    assert set(ev.id_unit) == set(split.loc[split.split == "evaluasi", "id_unit"])
    train = pd.read_csv(DATA / "rul_train.csv.gz", usecols=["id_unit", "siklus"])
    gagal = train.groupby("id_unit").siklus.max()
    np.testing.assert_array_equal(ev.aktual, (ev.id_unit.map(gagal) - ev.siklus).clip(upper=150))
    skor = metrik(ev.aktual, ev.prediksi)
    referensi = ringkas["model"][ringkas["final"]]["evaluasi"]
    for nama in ("rmse", "asimetris"):
        np.testing.assert_allclose(skor[nama], referensi[nama], rtol=1e-6)
    cakupan = ((ev.aktual >= ev.bawah) & (ev.aktual <= ev.atas)).mean()
    np.testing.assert_allclose(cakupan, ringkas["interval"]["cakupan"])
    assert sidik(DATA, BERKAS) == ringkas["hash"]
    assert len(PdfReader(pdf).pages) == 2
    print(f"Tugas 2 LULUS | {len(pred)} mesin uji | RMSE evaluasi {skor['rmse']:.6f} siklus")
    return {"status": "LULUS", "baris": len(pred), **skor, "cakupan": float(cakupan)}

## Sel 5 · Eksekusi Tugas 2

Jalankan jalankan_tugas2() setelah data siap. Checkpoint CPU yang cocok dapat dipakai kembali dari checkout repositori. Dataset tetap diperlukan untuk membentuk jendela, baseline, dan kalibrasi.

In [ ]:
from shutil import copy2
from threadpoolctl import threadpool_limits


def muat_checkpoint_referensi():
    lokasi = globals().get("REPO")
    if lokasi is None:
        kandidat = [Path.cwd(), *Path.cwd().parents]
        lokasi = next((p for p in kandidat if (p / "kode" / "tugas2" / "model").is_dir()), None)
    if lokasi is None or DEVICE.type != "cpu":
        return
    sumber = Path(lokasi) / "kode" / "tugas2" / "model"
    target = BUKTI / "model"
    target.mkdir(parents=True, exist_ok=True)
    for berkas in sumber.glob("*.pt"):
        keluaran = target / berkas.name
        if not keluaran.exists():
            copy2(berkas, keluaran)


def jalankan_tugas2():
    muat_checkpoint_referensi()
    torch.set_num_threads(3)
    with threadpool_limits(limits=3):
        jalankan(DATA, BUKTI, HASIL, DEVICE)
    ringkas = json.loads((BUKTI / "hasil.json").read_text(encoding="utf-8"))
    BAGIAN.parent.mkdir(parents=True, exist_ok=True)
    buat_bagian(HASIL, ringkas, BAGIAN)
    hasil = verifikasi_tugas2(ringkas, BAGIAN)
    simpan_json(BUKTI / "verifikasi.json", hasil)
    return hasil


hasil_tugas2 = jalankan_tugas2()

## Sel 6 · Ekspor berkas

Arsip memuat prediksi_rul.csv, PDF bagian, dan bukti eksperimen. Kolom interval tersimpan terpisah dari CSV penilaian.

In [ ]:
from zipfile import ZIP_DEFLATED, ZipFile

arsip = HASIL.parent / "hasil_tugas2.zip"
with ZipFile(arsip, "w", compression=ZIP_DEFLATED) as paket:
    for rel in ["prediksi_rul.csv", "laporan_bagian/02_prediksi_rul.pdf"]:
        paket.write(HASIL / rel, rel)
    for sumber in sorted(BUKTI.glob("*")):
        if sumber.is_file():
            paket.write(sumber, f"bukti_eksperimen/tugas2/{sumber.name}")
print("Arsip:", arsip)
if drive is not None:
    from google.colab import files
    files.download(str(arsip))